In [19]:
import requests
from bs4 import BeautifulSoup
import random
import pandas as pd
import re

In [20]:
title = "Data ML"
location = "Bagkok, Thailand"
start = 0  # Starting point for pagination

In [21]:
def split_criteria(criteria_array):
    criteria_dict = {}
    items = []
    for i in criteria_array:
        stripped = i.strip()
        if stripped != "":
            items.append(stripped)
    for index, item in enumerate(items):
        if index % 2 == 0:
            key = item
            value = items[index + 1]
            criteria_dict[key] = value
    return criteria_dict

In [22]:
# list_url = f"https://www.linkedin.com/jobs/search?keywords={title}&location={location}&pageNum=0".replace(" ", "%20")
list_url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords={title}&location={location}&start=0".replace(" ", "%2B")
response = requests.get(list_url)

list_data = response.text
list_soup = BeautifulSoup(list_data, 'html.parser')
page_jobs = list_soup.find_all('li')

In [23]:
print(list_url)

https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords=Data%2BML&location=Bagkok,%2BThailand&start=0


In [24]:
id_list = []
for job in page_jobs:
    base_card_div = job.find("div", {"class" : "base-card"})
    if base_card_div:
        # print(base_card_div)
        job_id = base_card_div.get("data-entity-urn", "").split(":")[-1]
        # print(f"Job ID: {job_id}")
        id_list.append(job_id)

# page_jobs[20].find("div", {"class" : "base-card"})

In [25]:
len(id_list)

10

In [26]:
print(f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{id_list[1]}")

https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/4300405780


In [27]:
job_list = []
for job_id in id_list:
    job_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"
    job_response = requests.get(job_url)
    job_post = {}
    job_soup = BeautifulSoup(job_response.text, 'html.parser')
    job_post['job_id'] = "li-" + job_id
    job_post['job_website'] = job_url.split(".")[1]
    job_post['job_url'] = job_url
    try:
        job_post['job_title'] = job_soup.find("h2", {"class" : "top-card-layout__title"}).text.strip()
    except:
        job_post['job_title'] = None
    try:
        job_post['company_name'] = job_soup.find("a", {"class": "topcard__org-name-link"}).text.strip()
    except:
        job_post['company_name'] = None
    try:
        job_post['company_url'] = job_soup.find("a", {"class": "topcard__org-name-link"})['href']
    except:
        job_post['company_url'] = None
    try:
        job_post['time_posted'] = job_soup.find("span", {"class" : "posted-time-ago__text"}).text.strip()
    except:
        job_post['time_posted'] = None
    try:
        job_post['num_applicants'] = job_soup.find("figcaption", {"class" : "num-applicants__caption"}).text.strip()
    except:
        try:
            job_post['num_applicants'] = job_soup.find("span", {"class" : "num-applicants__caption"}).text.strip()
        except:
            job_post['num_applicants'] = None
    try:
        job_post['location'] = job_soup.find("span", {"class" : "topcard__flavor topcard__flavor--bullet"}).text.strip()
    except:
        job_post['location'] = None
    try:
        job_criteria = job_soup.find("ul", {"class" : "description__job-criteria-list"}).text.strip().split("\n")
        criteria_dict = split_criteria(job_criteria)
        job_post['seniority_level'] = criteria_dict.get('Seniority level', None)
        job_post['employment_type'] = criteria_dict.get('Employment type', None)
        job_post['job_function'] = criteria_dict.get('Job function', None)
        job_post['industries'] = criteria_dict.get('Industries', None)
    except:
        job_post['seniority_level'] = None
        job_post['employment_type'] = None
        job_post['job_function'] = None
        job_post['industries'] = None
    try:
        description = job_soup.find("div", {"class" : "show-more-less-html__markup show-more-less-html__markup--clamp-after-5 relative overflow-hidden"})
        if description.find_all("strong"):
            for strong_tag in description.find_all("strong"):
                strong_tag.string = f"<strong>{strong_tag.text}</strong>"
        if description.find_all("ul"):
            for li_tag in description.find_all("li"):
                li_tag.string = f"<li>{li_tag.text}</li>"

        job_post['job_description'] = description.get_text(separator="\n").strip()
    except:
        job_post['job_description'] = None

    job_list.append(job_post)


In [28]:
job_list

[{'job_id': 'li-4262353945',
  'job_website': 'linkedin',
  'job_url': 'https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/4262353945',
  'job_title': 'Business Intelligence (Shopee) Fresh Graduate',
  'company_name': 'Shopee',
  'company_url': 'https://sg.linkedin.com/company/shopee?trk=public_jobs_topcard-org-name',
  'time_posted': '4 days ago',
  'num_applicants': 'Over 200 applicants',
  'location': 'Bangkok, Bangkok City, Thailand',
  'seniority_level': 'Mid-Senior level',
  'employment_type': 'Full-time',
  'job_function': 'Strategy/Planning, Analyst, and Consulting',
  'industries': 'Internet Marketplace Platforms and Technology, Information and Internet',
  'job_description': "<strong>Job Description</strong>\n<li>Developing reports, dashboards, analysis and some automations (will be generally with SQL, PYTHON, EXCEL, in-House tools)</li>\n<li>Assisting the various business units as a discussion partner to plan, strategise, and grow business by enabling data-driven decisi

In [29]:
# do dataframe stuff
df = pd.DataFrame(job_list)
df.head()

,job_id,job_website,job_url,job_title,company_name,company_url,time_posted,num_applicants,location,seniority_level,employment_type,job_function,industries,job_description
0,li-4262353945,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Business Intelligence (Shopee) Fresh Graduate,Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,4 days ago,Over 200 applicants,"Bangkok, Bangkok City, Thailand",Mid-Senior level,Full-time,"Strategy/Planning, Analyst, and Consulting","Internet Marketplace Platforms and Technology,...",<strong>Job Description</strong>\n<li>Developi...
1,li-4300405780,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,[TH] Business Intelligence - Fresh Graduate,Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,2 weeks ago,Over 200 applicants,"Bangkok, Bangkok City, Thailand",Entry level,Full-time,"Strategy/Planning, Analyst, and Consulting","Internet Marketplace Platforms and Technology,...",<strong>About The Team</strong>\nThe \n<strong...
2,li-4324559451,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Junior Business Analytics (ShopeePay),Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,5 days ago,182 applicants,"Bangkok, Bangkok City, Thailand",Entry level,Full-time,Business Development and Marketing,Financial Services,<strong>Job Description</strong>\nJob Descript...
3,li-4277459478,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Junior Marketing Analytics (ShopeePay),Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,7 hours ago,Over 200 applicants,"Bangkok, Bangkok City, Thailand",Entry level,Full-time,Business Development and Marketing,Financial Services,<strong>Job Description</strong>\n<li>Manage e...
4,li-4289428380,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Junior Business Analyst/Product Management (Sh...,Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,2 weeks ago,Over 200 applicants,"Bangkok, Bangkok City, Thailand",Entry level,Full-time,"Project Management, Strategy/Planning, and Pro...",Financial Services,<strong>Job Description</strong>\n<li>Gather a...


In [30]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime

def split_criteria(criteria_array):
    criteria_dict = {}
    items = []
    for i in criteria_array:
        stripped = i.strip()
        if stripped != "":
            items.append(stripped)
    for index, item in enumerate(items):
        if index % 2 == 0:
            key = item
            value = items[index + 1]
            criteria_dict[key] = value
    return criteria_dict

def request_preview_job_posting(title, location, i):
    list_url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords={title}&location={location}&start={i}".replace(" ", "%2B")
    try:
        response = requests.get(list_url)
    except:
        # sleep
        try:
            time.sleep(5)
            response = requests.get(list_url)
        except requests.exceptions.RequestException as e:
            print(f"Failed to retrieve data from {list_url}: {e}")
            return None
    list_data = response.text
    list_soup = BeautifulSoup(list_data, 'html.parser')
    page_jobs = list_soup.find_all('li')
    return page_jobs

def extract_job_ids(page_jobs):
    id_list = []
    for job in page_jobs:
        base_card_div = job.find("div", {"class" : "base-card"})
        if base_card_div:
            # print(base_card_div)
            job_id = base_card_div.get("data-entity-urn", "").split(":")[-1]
                # print(f"Job ID: {job_id}")
            id_list.append(job_id)
    return id_list

def scrape_job_posts(id_list):
    job_list = []
    for job_id in id_list:
        job_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"
        job_response = requests.get(job_url)
        job_post = {}
        job_soup = BeautifulSoup(job_response.text, 'html.parser')
        job_post['job_id'] = "li-" + job_id
        job_post['job_website'] = job_url.split(".")[1]
        job_post['job_url'] = job_url
        try:
            job_post['job_title'] = job_soup.find("h2", {"class" : "top-card-layout__title"}).text.strip()
        except:
            job_post['job_title'] = None
        try:
            job_post['company_name'] = job_soup.find("a", {"class": "topcard__org-name-link"}).text.strip()
        except:
            job_post['company_name'] = None
        try:
            job_post['company_url'] = job_soup.find("a", {"class": "topcard__org-name-link"})['href']
        except:
            job_post['company_url'] = None
        try:
            job_post['time_posted'] = job_soup.find("span", {"class" : "posted-time-ago__text"}).text.strip()
        except:
            job_post['time_posted'] = None
        try:
            job_post['num_applicants'] = job_soup.find("figcaption", {"class" : "num-applicants__caption"}).text.strip()
        except:
            try:
                job_post['num_applicants'] = job_soup.find("span", {"class" : "num-applicants__caption"}).text.strip()
            except:
                job_post['num_applicants'] = None
        try:
            job_post['location'] = job_soup.find("span", {"class" : "topcard__flavor topcard__flavor--bullet"}).text.strip()
        except:
            job_post['location'] = None
        try:
            job_criteria = job_soup.find("ul", {"class" : "description__job-criteria-list"}).text.strip().split("\n")
            criteria_dict = split_criteria(job_criteria)
            job_post['seniority_level'] = criteria_dict.get('Seniority level', None)
            job_post['employment_type'] = criteria_dict.get('Employment type', None)
            job_post['job_function'] = criteria_dict.get('Job function', None)
            job_post['industries'] = criteria_dict.get('Industries', None)
        except:
            job_post['experience_level'] = None
        try:
            description = job_soup.find("div", {"class" : "show-more-less-html__markup show-more-less-html__markup--clamp-after-5 relative overflow-hidden"})
            if description.find_all("strong"):
                for strong_tag in description.find_all("strong"):
                    strong_tag.string = f"<strong>{strong_tag.text}</strong>"
            if description.find_all("ul"):
                for li_tag in description.find_all("li"):
                    li_tag.string = f"<li>{li_tag.text}</li>"

            job_post['job_description'] = description.get_text(separator="\n").strip()
        except:
            job_post['job_description'] = None

        job_list.append(job_post)
    return job_list



# create main function (__main__ )
if __name__ == "__main__":
    now = datetime.now()
    timestamp = now.strftime("%Y-%m-%dT%H%M%S")

    title = "Data ML"
    location = "Bagkok, Thailand"
    job_list_all = []
    for i in range(0, 1000, 25):
        page_jobs = request_preview_job_posting(title, location, i)
        if page_jobs is None:
            break
        id_list = extract_job_ids(page_jobs)
        job_list = (scrape_job_posts(id_list))
        job_list_all.extend(job_list)
        print(f"Scraped {len(job_list_all)} job posts so far...")
        print(f"This is page starting at {i}")

    df = pd.DataFrame(job_list_all)
    #save to csv
    # df.to_csv(f"./raw/linkedin/{timestamp}.csv", index=False)
    # save ib json
    # df.to_json(f"./raw/linkedin/{timestamp}.json", orient="records", lines=True)

Scraped 10 job posts so far...
This is page starting at 0
Scraped 20 job posts so far...
This is page starting at 25
Scraped 30 job posts so far...
This is page starting at 50
Scraped 40 job posts so far...
This is page starting at 75
Scraped 50 job posts so far...
This is page starting at 100
Scraped 60 job posts so far...
This is page starting at 125
Scraped 70 job posts so far...
This is page starting at 150
Scraped 80 job posts so far...
This is page starting at 175
Scraped 90 job posts so far...
This is page starting at 200
Scraped 100 job posts so far...
This is page starting at 225
Scraped 110 job posts so far...
This is page starting at 250
Scraped 120 job posts so far...
This is page starting at 275
Scraped 130 job posts so far...
This is page starting at 300
Scraped 140 job posts so far...
This is page starting at 325
Scraped 150 job posts so far...
This is page starting at 350
Scraped 160 job posts so far...
This is page starting at 375
Scraped 170 job posts so far...
This i

In [34]:
df

,job_id,job_website,job_url,job_title,company_name,company_url,time_posted,num_applicants,location,seniority_level,employment_type,job_function,industries,job_description,experience_level
0,li-4262353945,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Business Intelligence (Shopee) Fresh Graduate,Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,4 days ago,Over 200 applicants,"Bangkok, Bangkok City, Thailand",Mid-Senior level,Full-time,"Strategy/Planning, Analyst, and Consulting","Internet Marketplace Platforms and Technology,...",<strong>Job Description</strong>\n<li>Developi...,NaN
1,li-4300405780,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,[TH] Business Intelligence - Fresh Graduate,Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,2 weeks ago,Over 200 applicants,"Bangkok, Bangkok City, Thailand",Entry level,Full-time,"Strategy/Planning, Analyst, and Consulting","Internet Marketplace Platforms and Technology,...",<strong>About The Team</strong>\nThe \n<strong...,NaN
2,li-4324559451,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Junior Business Analytics (ShopeePay),Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,5 days ago,184 applicants,"Bangkok, Bangkok City, Thailand",Entry level,Full-time,Business Development and Marketing,Financial Services,<strong>Job Description</strong>\nJob Descript...,NaN
3,li-4277459478,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Junior Marketing Analytics (ShopeePay),Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,7 hours ago,Over 200 applicants,"Bangkok, Bangkok City, Thailand",Entry level,Full-time,Business Development and Marketing,Financial Services,<strong>Job Description</strong>\n<li>Manage e...,NaN
4,li-4289428380,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Junior Business Analyst/Product Management (Sh...,Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,2 weeks ago,Over 200 applicants,"Bangkok, Bangkok City, Thailand",Entry level,Full-time,"Project Management, Strategy/Planning, and Pro...",Financial Services,<strong>Job Description</strong>\n<li>Gather a...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,li-4321524600,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Project Engineer,Filtrona,https://sg.linkedin.com/company/filtrona?trk=p...,1 week ago,116 applicants,"Bangkok City, Thailand",Mid-Senior level,Full-time,"Manufacturing, Analyst, and Project Management","Manufacturing, Industrial Machinery Manufactur...",<strong>Who We Are:</strong>\nFiltrona is the ...,NaN
376,li-4352583225,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Category Demand Planning Manager,Beiersdorf,https://de.linkedin.com/company/beiersdorf?trk...,3 days ago,167 applicants,"Bangkok, Bangkok City, Thailand",Mid-Senior level,Full-time,Management and Manufacturing,Personal Care Product Manufacturing,"Your Tasks\n<li> Developing accurate, high qua...",NaN
377,li-4366681675,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Sr. Medical Science Liaison,Bayer,https://de.linkedin.com/company/bayer?trk=publ...,1 week ago,28 applicants,"Bangkok, Bangkok City, Thailand",Mid-Senior level,Full-time,Research and Health Care Provider,"Pharmaceutical Manufacturing, Biotechnology Re...",<strong>Major Tasks </strong>\n<li>Exchange an...,NaN
378,li-4366418621,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Regional Strategic Accounts Lead(GSKA-EL),Lazada,https://sg.linkedin.com/company/lazada?trk=pub...,1 week ago,73 applicants,"Bangkok, Bangkok City, Thailand",Mid-Senior level,Full-time,"Business Development, Project Management, and ...",Internet Marketplace Platforms and IT Services...,The Regional Strategic Account Lead role is re...,NaN


In [36]:
df['job_title'].value_counts()

job_title
Data Engineer                                                       7
AI Engineer                                                         4
Business Analyst                                                    3
Marketing Manager                                                   3
Media Planner                                                       3
                                                                   ..
Data Scientist, KYC Anti Fraud                                      1
เจ้าหน้าที่วิเคราะห์ข้อมูล (Data Analysis Statistic)                1
Shopee's Future Operations Leaders Program (FLP) 2026 (Thailand)    1
eRTM Marketing Strategy Support Manager                             1
Vice President - Merchandising (Budget Control & Report)            1
Name: count, Length: 328, dtype: int64

In [37]:
# import linkedin job posts from csv
df_check = pd.read_csv("../raw/linkedin/2026-02-06T015239.csv")


In [40]:
# check duplicate df_check and df by join and check uqinue
df_check.merge(df, on='job_id', how='inner')

,job_id,job_website_x,job_url_x,job_title_x,company_name_x,company_url_x,time_posted_x,num_applicants_x,location_x,seniority_level_x,...,company_url_y,time_posted_y,num_applicants_y,location_y,seniority_level_y,employment_type_y,job_function_y,industries_y,job_description_y,experience_level_y
0,li-4262353945,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Business Intelligence (Shopee) Fresh Graduate,Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,4 days ago,Over 200 applicants,"Bangkok, Bangkok City, Thailand",Mid-Senior level,...,https://sg.linkedin.com/company/shopee?trk=pub...,4 days ago,Over 200 applicants,"Bangkok, Bangkok City, Thailand",Mid-Senior level,Full-time,"Strategy/Planning, Analyst, and Consulting","Internet Marketplace Platforms and Technology,...",<strong>Job Description</strong>\n<li>Developi...,NaN
1,li-4300405780,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,[TH] Business Intelligence - Fresh Graduate,Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,2 weeks ago,Over 200 applicants,"Bangkok, Bangkok City, Thailand",Entry level,...,https://sg.linkedin.com/company/shopee?trk=pub...,2 weeks ago,Over 200 applicants,"Bangkok, Bangkok City, Thailand",Entry level,Full-time,"Strategy/Planning, Analyst, and Consulting","Internet Marketplace Platforms and Technology,...",<strong>About The Team</strong>\nThe \n<strong...,NaN
2,li-4369104006,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Data Engineer,OxygenAI,https://th.linkedin.com/company/oxygenai?trk=p...,NaN,Be among the first 25 applicants,Bangkok Metropolitan Area,Entry level,...,https://th.linkedin.com/company/oxygenai?trk=p...,7 hours ago,Be among the first 25 applicants,Bangkok Metropolitan Area,Entry level,Full-time,Information Technology,Software Development,<strong>Responsibilities:</strong>\n<li>Design...,NaN
3,li-4198762497,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Data Engineer,"Locus Telecommunication Inc., Ltd.",https://th.linkedin.com/company/locus-telecomm...,10 months ago,112 applicants,"Bangkok, Bangkok City, Thailand",Entry level,...,https://th.linkedin.com/company/locus-telecomm...,10 months ago,112 applicants,"Bangkok, Bangkok City, Thailand",Entry level,Full-time,Information Technology,Software Development,<strong>Job Description</strong>\n<li>Work on ...,NaN
4,li-4321427106,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Data Engineer (IT),DHL eCommerce,https://de.linkedin.com/company/dhlecommerce?t...,1 week ago,76 applicants,"Bangkok, Bangkok City, Thailand",Mid-Senior level,...,https://de.linkedin.com/company/dhlecommerce?t...,1 week ago,76 applicants,"Bangkok, Bangkok City, Thailand",Mid-Senior level,Full-time,Information Technology,Truck Transportation,Welcome to DHL eCommerce!\nWe are excited to a...,NaN
5,li-4367838440,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Data Engineer,Inteltion,https://th.linkedin.com/company/inteltion?trk=...,3 days ago,83 applicants,"Bangkok, Bangkok City, Thailand",Entry level,...,https://th.linkedin.com/company/inteltion?trk=...,3 days ago,83 applicants,"Bangkok, Bangkok City, Thailand",Entry level,Full-time,Information Technology,IT Services and IT Consulting,<strong>About the Company</strong>\nAt Intelti...,NaN
6,li-4324559451,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,Junior Business Analytics (ShopeePay),Shopee,https://sg.linkedin.com/company/shopee?trk=pub...,5 days ago,183 applicants,"Bangkok, Bangkok City, Thailand",Entry level,...,https://sg.linkedin.com/company/shopee?trk=pub...,5 days ago,184 applicants,"Bangkok, Bangkok City, Thailand",Entry level,Full-time,Business Development and Marketing,Financial Services,<strong>Job Description</strong>\nJob Descript...,NaN
7,li-4340303441,linkedin,https://www.linkedin.com/jobs-guest/jobs/api/j...,AI Engineer and Data Specialist,ooca - อูก้า แอปฯจิตวิทยา,https://th.linkedin.com/company/ooca?trk=publi...,2 months ago,Be among the first 25 applican